# NB1 — SFT-mini: mô hình SFT tiếng Việt (checkpoint) làm điểm xuất phát cho DPO

**Công nghệ:** Unsloth + LoRA r=16 trên Qwen3-4B-Instruct-2507 (4-bit) + 1k VN Alpaca, 1 epoch.

> **Mục tiêu:** tạo mô hình SFT để DPO align tiếp. Notebook lưu hai thứ:
> - `adapters/sft-mini/`: LoRA adapter (nhẹ, để nộp bài);
> - `models/sft-merged/`: SFT đã gộp vào trọng số 16-bit. **NB3 huấn luyện DPO trên mô hình này**,
>   nên reference của DPO chính là mô hình SFT (lab cũ dùng nhầm mô hình gốc làm reference).
>
> Mô hình gốc đã là bản instruct, nên SFT ở đây chủ yếu kéo phong cách trả lời về
> tiếng Việt kiểu Alpaca. Loss chỉ tính trên phần trả lời (`train_on_responses_only`).

In [1]:
import sys
import json
import time
from pathlib import Path

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "lab22" / "config.py").exists())
sys.path.insert(0, str(ROOT))

import unsloth  # noqa: F401  (must load before trl/transformers)
import torch

from lab22 import config as C
from lab22 import modeling as MD

assert torch.cuda.is_available(), "NB1 needs a CUDA GPU. See HARDWARE-GUIDE.md."
C.ensure_dirs()
print(C.summary())

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/opt/day22-venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


🦥 Unsloth Zoo will now patch everything to make training faster!


COMPUTE_TIER=T4 base=Qwen/Qwen3-0.6B max_len=768 seed=42
SFT saillab/alpaca-vietnamese-cleaned[:1000]  PREF sailor2/sea-ultrafeedback-onpolicy (Vietnamese) train=800 eval=100
DPO beta=0.1 lr=5e-06 epochs=1.0 loss=['sigmoid']


## 1. Nạp mô hình 4-bit + LoRA

In [2]:
model, tokenizer = MD.load_model(C.BASE_MODEL)
model = MD.add_lora(model)
print(f"pad={tokenizer.pad_token!r} eos={tokenizer.eos_token!r}")
print(f"Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

==((====))==  Unsloth 2026.10.2: Fast Qwen3 patching. Transformers: 5.17.0.
   \\   /|    NVIDIA GeForce RTX 3050 Ti Laptop GPU. Num GPUs = 1. Max memory: 4.0 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.7.0+cu118. CUDA: 8.6. CUDA Toolkit: 11.8. Triton: 3.3.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.30. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

Loading weights:   0%|          | 1/310 [00:14<1:16:12, 14.80s/it]

Loading weights:   1%|          | 3/310 [00:14<19:58,  3.90s/it]  

Loading weights:   2%|▏         | 5/310 [00:15<10:54,  2.15s/it]

...

Loading weights:  99%|█████████▉| 308/310 [00:32<00:00, 20.78it/s]

Loading weights: 100%|██████████| 310/310 [00:32<00:00,  9.61it/s]

[unsloth.import_fixes|WARNING]Unsloth: Ignoring an unusable torchao so LoRA can still be built (Found an incompatible version of torchao. Found version 0.13.0, but only versions above 0.16.0 are supported). Run `pip install --upgrade torchao` if you need torchao quantization.


Unsloth 2026.10.2 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


pad='<|vision_pad|>' eos='<|im_end|>'
Trainable params: 10,092,544


## 2. VN Alpaca → chat text

In [3]:
from datasets import load_dataset

ds = load_dataset(C.SFT_DATASET, split=f"train[:{C.SFT_SLICE}]")
print(f"Loaded {len(ds)} rows. Columns: {ds.column_names}")


def to_text(row):
    # The translated rows carry stray leading spaces; strip so the template stays clean.
    instruction, extra = row["instruction"].strip(), (row.get("input") or "").strip()
    prompt = instruction + (f"\n\n{extra}" if extra else "")
    messages = [
        {"role": "user", "content": prompt},
        {"role": "assistant", "content": row["output"].strip()},
    ]
    return {"text": MD.chat_text(tokenizer, messages, add_generation_prompt=False)}


ds = ds.filter(lambda r: bool(r.get("instruction")) and bool(r.get("output")))
ds = ds.map(to_text, remove_columns=ds.column_names)
print(ds[0]["text"][:400])

Generating train split:   0%|          | 0/41601 [00:00<?, ? examples/s]

Generating train split:   2%|▏         | 1000/41601 [00:01<01:04, 626.85 examples/s]

Generating train split:   5%|▍         | 2000/41601 [00:01<00:28, 1375.56 examples/s]

Generating train split:  12%|█▏        | 5000/41601 [00:01<00:09, 4016.81 examples/s]

Generating train split:  26%|██▋       | 11000/41601 [00:02<00:02, 10524.14 examples/s]

...

Map:  26%|██▌       | 259/1000 [00:00<00:00, 2524.38 examples/s]

Map:  63%|██████▎   | 627/1000 [00:00<00:00, 3195.54 examples/s]

Map: 100%|██████████| 1000/1000 [00:00<00:00, 2256.50 examples/s]

Map: 100%|██████████| 1000/1000 [00:00<00:00, 2188.94 examples/s]

<|im_start|>user
Đưa ra một mô tả về một con vật, xác định loại động vật đó là gì.

Loài vật này có chiếc cổ dài, linh hoạt và mỏ hẹp, nhọn. Nó có chân dài và bàn chân có màng, và nó ăn chủ yếu là cá.<|im_end|>
<|im_start|>assistant
<think>

</think>

Con vật được mô tả có khả năng là một con diệc.<|im_end|>



## 3. Huấn luyện (loss chỉ trên phần trả lời)

In [4]:
from trl import SFTConfig, SFTTrainer
from unsloth.chat_templates import train_on_responses_only

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=ds,
    args=SFTConfig(
        output_dir=str(C.ADAPTERS / "sft-checkpoints"),
        dataset_text_field="text",
        max_length=C.MAX_LEN,
        per_device_train_batch_size=C.TIER.sft_batch,
        gradient_accumulation_steps=C.TIER.sft_grad_accum,
        num_train_epochs=1,
        learning_rate=2e-4,
        lr_scheduler_type="cosine",
        warmup_steps=0.03,
        logging_steps=10,
        save_strategy="no",
        optim="adamw_8bit",
        seed=C.SEED,
        report_to="none",
        **MD.precision_flags(),
    ),
)
# Qwen ChatML markers; prompt tokens get label -100.
trainer = train_on_responses_only(
    trainer,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)
supervised_tokens = sum(sum(label != -100 for label in row["labels"]) for row in trainer.train_dataset)
total_tokens = sum(len(row["labels"]) for row in trainer.train_dataset)
assert 0 < supervised_tokens < total_tokens
torch.cuda.reset_peak_memory_stats()
training_started = time.perf_counter()
result = trainer.train()
train_seconds = time.perf_counter() - training_started
print(f"Final SFT loss: {result.training_loss:.4f}")

Unsloth: reducing dataset_num_proc 8 -> 2 to fit free memory (~1GB per worker). Set UNSLOTH_DATASET_NUM_PROC to override.


Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/1000 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=2):  50%|█████     | 500/1000 [00:04<00:04, 105.08 examples/s]

Unsloth: Tokenizing ["text"] (num_proc=2): 100%|██████████| 1000/1000 [00:08<00:00, 119.68 examples/s]

Unsloth: Tokenizing ["text"] (num_proc=2): 100%|██████████| 1000/1000 [00:08<00:00, 115.17 examples/s]

...

Map: 100%|██████████| 1000/1000 [00:00<00:00, 4368.10 examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,000 | Num Epochs = 1 | Total steps = 125
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 10,092,544 of 606,142,464 (1.67% trained)


Step,Training Loss
10,2.047258
20,1.877671
30,1.970041
40,1.841197
50,1.832886
60,1.965000
70,1.803286
80,1.812480
90,1.792272
100,1.781947


Final SFT loss: 1.8555


In [5]:
import matplotlib.pyplot as plt
import pandas as pd

logs = pd.DataFrame([r for r in trainer.state.log_history if "loss" in r])
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.plot(logs["step"], logs["loss"], color="#2e548a")
ax.set_xlabel("step")
ax.set_ylabel("SFT loss (response tokens)")
ax.set_title(f"SFT-mini · {C.COMPUTE_TIER} · {C.BASE_MODEL.split('/')[-1]}")
ax.grid(True, alpha=0.3)
C.SCREENSHOTS.mkdir(parents=True, exist_ok=True)
fig.savefig(C.SCREENSHOTS / "02-sft-loss.png", dpi=120, bbox_inches="tight")
plt.show()

## 4. Lưu adapter + mô hình SFT đã gộp

In [6]:
model.save_pretrained(str(C.SFT_ADAPTER))
tokenizer.save_pretrained(str(C.SFT_ADAPTER))
model.save_pretrained_merged(str(C.SFT_MERGED), tokenizer, save_method="merged_16bit")
metrics = {
    "base_model": C.BASE_MODEL,
    "dataset": C.SFT_DATASET,
    "samples": len(ds),
    "max_length": C.MAX_LEN,
    "seed": C.SEED,
    "epochs": 1,
    "train_seconds": train_seconds,
    "peak_vram_gb": torch.cuda.max_memory_allocated() / 1024**3,
    "mean_train_loss": float(result.training_loss),
    "optimizer_steps": trainer.state.global_step,
    "supervised_tokens": supervised_tokens,
    "total_tokens": total_tokens,
    "supervised_fraction": supervised_tokens / total_tokens,
    "log_history": trainer.state.log_history,
}
(C.SFT_ADAPTER / "sft_metrics.json").write_text(json.dumps(metrics, indent=2), encoding="utf-8")
print(f"Saved adapter → {C.SFT_ADAPTER}\nSaved merged 16-bit → {C.SFT_MERGED}")

Unsloth: Restored added_tokens_decoder metadata in /lab/adapters/sft-mini/tokenizer_config.json.


Unsloth: Downloading `unsloth/Qwen3-0.6B` into the Hugging Face cache so future exports skip the 1.1GB download...


Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

Fetching 1 files: 100%|██████████| 1/1 [02:00<00:00, 120.36s/it]

Fetching 1 files: 100%|██████████| 1/1 [02:00<00:00, 120.37s/it]

...

Unsloth: Merging weights into 16bit:   0%|          | 0/1 [00:00<?, ?it/s]

Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [02:39<00:00, 159.04s/it]

Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [02:39<00:00, 159.04s/it]

Unsloth: Merge process complete. Saved to `/lab/models/sft-merged`


Saved adapter → /lab/adapters/sft-mini
Saved merged 16-bit → /lab/models/sft-merged


In [7]:
sample = MD.generate(model, tokenizer, ["Giải thích ngắn gọn (3-4 câu) thuật toán quicksort hoạt động thế nào."], 200)
print(sample[0])

Both `max_new_tokens` (=200) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Thuật toán quicksort là một thuật toán sắp xếp sắp xếp các phần tử trong một danh sách theo thứ tự tăng dần hoặc giảm dần. Nó hoạt động bằng cách chọn một phần tử đầu tiên trong danh sách, gọi là pivot, và sắp xếp các phần tử còn lại theo thứ tự tăng dần hoặc giảm dần theo cách mà pivot được đặt ở vị trí trung tâm. Sau đó, các phần tử còn lại được sắp xếp theo thứ tự tăng dần hoặc giảm dần theo thứ tự của pivot, và các phần tử còn lại được sắp xếp theo thứ tự tăng dần hoặc giảm dần theo thứ tự của pivot.


## 5. Ghi chú vibe-coding

> **Cần bao nhiêu SFT để DPO có ý nghĩa?** Thử `SFT_SLICE=100` rồi chạy lại NB1 → NB3.
> Margin ở NB3 còn tăng không? Câu trả lời còn mạch lạc không? Ghi giả thuyết trước
> khi chạy, kết quả vào `submission/REFLECTION.md` §6.

**Tiếp theo:** NB2 — dữ liệu sở thích (preference) tiếng Việt.